In [11]:
# import env file, make sure it contains your kaggle token

from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())   # finds ../.env from notebooks/
print("token loaded:", "KAGGLE_API_TOKEN" in os.environ)

token loaded: True


In [2]:
# download dataset

import kagglehub
from pathlib import Path

DATA_FOLDER = "./../data/raw/"

# Download latest version
if not Path(DATA_FOLDER).exists():
    path = kagglehub.competition_download('favorita-grocery-sales-forecasting', 
                                      output_dir=DATA_FOLDER)

    print("Path to competition files:", path)

In [4]:
# extract files

from pathlib import Path
import py7zr

EXTRACTED_FOLDER = Path(DATA_FOLDER) / "extracted"

if not Path(EXTRACTED_FOLDER):
    folder = Path(DATA_FOLDER)
    for archive in sorted(folder.glob("*.csv.7z")):
        out_dir = archive.with_suffix("")
        print(f"Extracting {archive.name} -> {out_dir.name}")
        try:
            with py7zr.SevenZipFile(archive, mode="r") as z:
                z.extractall(path=EXTRACTED_FOLDER)
        except Exception as e:
            print(f"  Failed: {e}")


In [ ]:
# convert to parquet

import polars as pl

PROCESSED_FOLDER = Path("./../data/processed/")

(
    pl.scan_csv(
        EXTRACTED_FOLDER / "train.csv",
        schema_overrides={
            "store_nbr": pl.Int8,# 1 to 54
            "item_nbr": pl.Int32, # up to ~2.1 million
            "unit_sales": pl.Float32, # decimals
            "onpromotion": pl.Boolean, # True / False / empty
        },
        try_parse_dates=True
    )
    .drop("id")
    .sink_parquet(PROCESSED_FOLDER / "train.parquet", mkdir=True)
)